# Job Recommendation – TF‑IDF Improved (Option 2)

* Tổng quan

- Xây dựng baseline Job Recommendation dựa trên TF-IDF

- Luồng xử lý:

    - Biểu diễn văn bản bằng TF-IDF

    - So khớp CV – Job bằng cosine similarity trên TF-IDF vector

- Mục tiêu:

    - Tạo tập job ứng viên ban đầu

    - Làm baseline để so sánh với phiên bản cải tiến (taxonomy, rerank, explanation)

- Pipeline tổng quát

    - Load dữ liệu CV và Job

    - Chuẩn hoá văn bản

    - Vector hoá văn bản bằng TF-IDF

    - Truy hồi (retrieval) các job phù hợp với CV

    - Trả về top-K job baseline


In [18]:
# ===== Paths (edit if needed) =====
jobs_path = "/Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/datasets/jobs_merged_it.xlsx"
cvs_path  = "/Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/datasets/UpdatedResumeDataSet.csv"

# ===== Config =====
TOPN_RETRIEVE = 200     # lọc sơ ra 200 job giống CV nhất về mặt chữ trước - giống nhất
TOPK_FINAL    = 10      # trả về 10 job tốt nhất cho ứng viên xem

# Weights for final_score
W_SEM   = 0.60      # So chữ (TF-IDF)
W_SKILL = 0.25      # Nếu kỹ năng trùng nhiều
W_TAX   = 0.15      # Job đúng ngành (backend với backend)
W_SEN_P = 0.30      # dựa vào mức độ junior / mid / senior giữa CV và job (KINH NGHIỆM)


In [19]:
import numpy as np
import pandas as pd
import re

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


## Bước 1: Load data và kiểm tra dữ liệu

* Mục đích: Chuẩn bị dữ liệu đầu vào cho toàn bộ pipeline

- Load các file dữ liệu:

    - CV / Resume

    - Job description

- Kiểm tra:

    - Số lượng CV và Job

    - Các cột văn bản chính (title, description, skills, …)

- Đảm bảo dữ liệu:

    - Không bị thiếu trường quan trọng

    - Định dạng sẵn sàng cho xử lý text

In [27]:
jobs = pd.read_excel(jobs_path)
cvs  = pd.read_csv(cvs_path)

print("Jobs shape:", jobs.shape)
print("CVs shape:", cvs.shape)
print("Jobs columns:", list(jobs.columns))
print("CVs columns:", list(cvs.columns))

# ---- minimal text cleaning (safe, fast) ----
def clean_text(s: str) -> str:
    s = str(s) if s is not None else ""
    s = s.lower()
    s = re.sub(r"\s+", " ", s).strip()
    return s

# ---- schema adapter for merged jobs dataset ----
# jobs_merged_it.xlsx typically has: title, description, company, location, source
# notebook expects: title, cleaned_description

# 1) ensure required source columns exist
for col in ["title", "description"]:
    if col not in jobs.columns:
        raise ValueError(f"jobs dataset missing required column: {col}")

# 2) create cleaned_description (from description) if missing
if "cleaned_description" not in jobs.columns:
    jobs["cleaned_description"] = jobs["description"]

# 3) fill NaN + cast to string
for col in ["title", "cleaned_description", "company", "location", "source"]:
    if col in jobs.columns:
        jobs[col] = jobs[col].fillna("").astype(str)

# 4) (optional but recommended) clean text fields
jobs["title"] = jobs["title"].apply(clean_text)
jobs["cleaned_description"] = jobs["cleaned_description"].apply(clean_text)

# 5) optional: build a single job_text used downstream (if your notebook uses it)
# If your notebook already builds job_text later, you can skip this.
jobs["job_text"] = jobs["title"] + " " + jobs["cleaned_description"]

# ---- final required-column check (same as your original logic) ----
for col in ["title", "cleaned_description"]:
    if col not in jobs.columns:
        raise ValueError(f"jobs dataset missing required column: {col}")

# ---- quick preview ----
display(jobs.head(3))
display(cvs.head(3))

resumes = cvs                # some functions use `resumes` name
job_texts = jobs["job_text"] # Step 2 expects this
cv_texts  = resumes["Resume"]  # Step 2 expects this (UpdatedResumeDataSet has Resume)

print("job_texts:", job_texts.shape, "cv_texts:", cv_texts.shape)

Jobs shape: (89277, 5)
CVs shape: (962, 2)
Jobs columns: ['title', 'description', 'company', 'location', 'source']
CVs columns: ['Category', 'Resume']


,title,description,company,location,source,cleaned_description,job_text
0,support technologist ii 2024-02216,**description and functions** ----------------...,State of Wyoming,"Rock Springs, WY",it,**description and functions** ----------------...,support technologist ii 2024-02216 **descripti...
1,computer technician,***if you would like to apply for this positio...,Medford Township Public Schools,"Medford, NJ",it,***if you would like to apply for this positio...,computer technician ***if you would like to ap...
2,it technician,i need a sped up recording converted to text. ...,Law Office of Ruben and Ruben LLC,,it,i need a sped up recording converted to text. ...,it technician i need a sped up recording conve...


,Category,Resume
0,Data Science,Skills * Programming Languages: Python (pandas...
1,Data Science,Education Details \r\nMay 2013 to May 2017 B.E...
2,Data Science,"Areas of Interest Deep Learning, Control Syste..."


job_texts: (89277,) cv_texts: (962,)


## Bước 2: Chuẩn hoá text

- Chuyển hết về chữ thường

- Xoá ký tự rác, nhưng giữ lại: + ; # ; .

- Dọn khoảng trắng

- Xoá khoảng trắng thừa đầu/cuối

In [21]:
# ===== Step 2: Normalize text =====

def norm_text(t: str) -> str:
    t = str(t).lower()
    t = re.sub(r"[^a-z0-9\s\+\#\.]", " ", t)   # keep + # . for c++, c#, node.js
    t = re.sub(r"\s+", " ", t)
    return t.strip()

job_texts_norm = job_texts.apply(norm_text)  # lấy từng job đã được chuẩn hoá chữ
cv_texts_norm  = cv_texts.apply(norm_text)   # lấy từng CV đã được chuẩn hoá chữ


## Bước 3: Chuẩn hoá vector từ chữ sang số

- Mục đích: Biến văn bản thành vector số để có thể tính toán độ giống nhau

- Khởi tạo TF-IDF Vectorizer

- Biến đổi:

    - CV text → TF-IDF vector

    - Job text → TF-IDF vector

- Kết quả:

    - Mỗi CV / Job là một vector trong không gian TF-IDF

In [22]:
# ===== Step 3: TF-IDF vectorization =====
tfidf = TfidfVectorizer(
    max_features=30000,     # đếm chữ và chỉ nhớ tối đa 30.000 từ/cụm từ quan trọng nhất
    ngram_range=(1,2),      # 1 chữ đơn và 2 chữ liền
    min_df=3,               # chữ nào xuất hiện dưới 3 job => bỏ
    stop_words="english"    # bỏ những từ vô nghĩa như: the, and, is, for,...
)

job_tfidf = tfidf.fit_transform(job_texts_norm) # đọc hết chữ all job, học xem chữ nào quan trọng
                                                # rồi biến mỗi job thành 1 dãy số
print("job_tfidf shape:", job_tfidf.shape)


job_tfidf shape: (21961, 30000)


## Bước 4: Retrieval (lọc job ứng viên ban đầu)

- Mục đích: Tìm các job giống CV nhất về mặt nội dung chữ

- Với mỗi CV:

    - Tính cosine similarity giữa vector CV và tất cả vector Job

    - Sắp xếp job theo độ giống giảm dần

- Lấy:

    - Top-N job để làm candidate set ban đầu

In [23]:
# trả về top N job giống CV nhất (dựa trên chữ)
def retrieve_topn_jobs_for_cv_tfidf(cv_index: int, topn: int = TOPN_RETRIEVE):
    cv_vec = tfidf.transform([cv_texts_norm.iloc[cv_index]])   # CV đã được chuyển dãy số
    sims = cosine_similarity(cv_vec, job_tfidf)[0]  # độ tương đồng giữa job và CV
    idx = np.argsort(-sims)[:topn] # sắp xếp simil giảm dần

    cands = jobs.iloc[idx].copy() # Lấy thông tin job tương ứng với topN index
    cands["semantic_score"] = sims[idx] # Cột được thêm để biết job giống CV như nào
    return cands, idx


## Baseline Top-K Recommendation

- Mục đích: Trả về kết quả gợi ý job cuối cùng theo TF-IDF thuần

- Hàm baseline_topk_tfidf:

    - Input:

        - cv_index: CV cần recommend

        - top_k: số job trả về

        - topn_retrieve: số job lấy ở bước retrieval

    - Xử lý:

        - Tính similarity

        - Lọc top job phù hợp nhất

    - Output:

        - DataFrame chứa:

            - Job ID

            - Job title

            - Similarity score

In [24]:
def baseline_topk_tfidf(cv_index: int, top_k: int = TOPK_FINAL, topn_retrieve: int = TOPN_RETRIEVE) -> pd.DataFrame:
    cands, _ = retrieve_topn_jobs_for_cv_tfidf(cv_index, topn_retrieve)
    b = cands.sort_values("semantic_score", ascending=False).head(top_k).copy()
    return b[["title", "company", "location", "semantic_score"]]


### Diễn giải semantic_score

semantic_score được tính bằng cosine similarity giữa biểu diễn văn bản của hồ sơ ứng viên (CV) và nội dung mô tả của vị trí công việc. Do cosine similarity được áp dụng trong không gian vector có chiều cao và trên các văn bản dài, các giá trị similarity thu được thường không lớn về mặt tuyệt đối. Vì vậy, việc diễn giải semantic_score nên dựa trên các khoảng giá trị tương đối.

Bảng dưới đây trình bày cách diễn giải định tính các giá trị semantic_score trong mô hình baseline:

| Khoảng semantic_score | Ý nghĩa                                   |
|----------------------|--------------------------------------------|
| < 0.05               | Gần như không liên quan                    |
| 0.05 – 0.10          | Liên quan rất yếu                          |
| 0.10 – 0.15          | Liên quan ở mức trung bình                 |
| **0.15 – 0.25**      | **Liên quan tốt (baseline)**               |
| > 0.25               | Liên quan rất mạnh (hiếm gặp với baseline) |

Trên thực tế, phần lớn các gợi ý job hợp lý trong mô hình baseline nằm trong khoảng từ 0.15 đến 0.25, đây là hiện tượng bình thường và phù hợp với đặc tính của cosine similarity.


In [25]:
# ===== Demo TF-IDF baseline =====
demo_idx = 0
print("Demo CV index:", demo_idx)
display(baseline_topk_tfidf(demo_idx, top_k=10))


Demo CV index: 0


,title,company,location,semantic_score
14341,it deskside support,Infinite Computing Systems,"Cedar Rapids, IA, US",0.183222
6810,avp - fraud risk manager,Barclays,"Henderson, NV, US",0.180198
20892,product owner i,GM Financial,"Irving, TX, US",0.180054
14568,sales and technical support representative,Computer Corner,"Oshkosh, WI, US",0.172438
19214,it support specialist,Vintage Realty Company,"Shreveport, LA, US",0.166611
20744,it operations & support analyst i - its,State of Idaho,"Boise, ID, US",0.162500
1349,director of data analytics and business intell...,Central Health,"Austin, TX, US",0.162112
18932,information security analyst iii,Clinical Reference Laboratory,"Lenexa, KS, US",0.158973
18230,508 certified accessibility/penetration tester...,OMG Technology,"NJ, US",0.158563
12839,product associate,JPMorganChase,"New York, NY, US",0.156674
